### 데이터셋 전처리

In [ ]:
import pandas as pd
from datasets import Dataset, Audio
from transformers import WhisperProcessor
import os

In [ ]:
# 경로 및 설정 상수
CACHE_DIR = "./audio_cache"
TRAIN_SAVE_PATH = "train_dataset"
EVAL_SAVE_PATH = "eval_dataset"
TEST_SPLIT = 0.2
RANDOM_SEED = 42

# 데이터 로드
data = pd.read_csv("whisper_finetuning_data_ds_5.csv")  # 'audio_path', 'transcription' 포함
dataset = Dataset.from_pandas(data)
dataset = dataset.cast_column("audio_path", Audio(cache_dir=CACHE_DIR))  # 오디오 캐싱 설정

# Whisper 프로세서 로드
model_name = "openai/whisper-medium"
processor = WhisperProcessor.from_pretrained(model_name)

# 데이터 전처리 함수
def preprocess_data(examples):
    input_features = []
    labels = []

    for audio_path, transcription in zip(examples["audio_path"], examples["transcription"]):
        try:
            audio = audio_path  # 오디오 데이터 로드
            if not audio or "array" not in audio or "sampling_rate" not in audio:
                print(f"오디오 파일이 제대로 로드되지 않았습니다. 건너뛰기: {audio_path}")
                continue

            # 오디오 데이터 및 텍스트 전처리
            inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").input_features
            label = processor(text=transcription, return_tensors="pt").input_ids

            input_features.append(inputs.squeeze())
            labels.append(label.squeeze())
        except Exception as e:
            print(f"오류 발생: {e}. 건너뛰기: {audio_path}")
            continue

    return {"input_features": input_features, "labels": labels}

# 데이터셋 전처리
print("데이터셋 전처리 중...")
dataset = dataset.map(preprocess_data, remove_columns=["audio_path", "transcription"], batched=True, batch_size=32)

# None 값 필터링 및 데이터 손실 추적
initial_len = len(dataset)
dataset = dataset.filter(lambda example: example["input_features"] is not None and example["labels"] is not None)
filtered_len = len(dataset)
print(f"필터링 완료: {initial_len - filtered_len}개의 데이터가 제거되었습니다.")

# 데이터셋 분리
train_test_split = dataset.train_test_split(test_size=TEST_SPLIT, seed=RANDOM_SEED)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

# 전처리된 데이터 저장
train_dataset.save_to_disk(TRAIN_SAVE_PATH)
eval_dataset.save_to_disk(EVAL_SAVE_PATH)
print("전처리된 데이터셋 저장 완료")

### 파인튜닝

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Trainer, TrainingArguments
from datasets import load_from_disk
from datasets import load_metric
import torch
from typing import Any, Dict, List

In [ ]:
# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Whisper 모델 및 프로세서 불러오기
model_name = "openai/whisper-medium"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)

# 전처리된 데이터 로드
train_dataset = load_from_disk("train_dataset")
eval_dataset = load_from_disk("eval_dataset")

# Word Error Rate (WER) 계산을 위한 평가 함수
wer_metric = load_metric("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# 데이터 Collator
class WhisperDataCollator:
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = [torch.tensor(feature["input_features"]) for feature in features]
        input_features = torch.stack(input_features)
        labels = torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(feature["labels"]) for feature in features],
            batch_first=True,
            padding_value=-100
        )
        return {"input_features": input_features, "labels": labels}

data_collator = WhisperDataCollator()

# 학습 파라미터 설정
training_args = TrainingArguments(
    output_dir="./whisper_finetuning",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    evaluation_strategy="epoch",
    num_train_epochs=3,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=500,
    save_steps=1000,
    save_total_limit=2,
    learning_rate=5e-5,
    fp16=True
)

# Trainer 설정
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 파인튜닝 진행
print("Whisper 모델 파인튜닝 시작...")
trainer.train()

# 모델 저장
model.save_pretrained("./whisper_finetuned_model")
print("모델 파인튜닝 완료 및 저장 완료")